# UAB · Day 2: Balances, dynamics & control

**Worked solutions.**

Build a tank balance, add P and PI control, then solve the coupled oxygen-limited chemostat.

Deliver tank and chemostat simulations with a justified solver and controller sign.

Time is in hours unless specified; model concentrations are g/L and volume is L.

Source: YeastFermentation Days 1–3. Python translation: Johan Le Nepvou De Carfort. Original MATLAB teaching material: Pedram Ramin and Krist Gernaey, PROSYS, DTU.

Run the embedded support-file setup once. For local Jupyter install numpy, scipy, and matplotlib first.


In [ ]:
# These support files are embedded for portable, offline notebook use.
# Original files retain their source content; course_models.py documents adapters.
import json
from pathlib import Path
_support = json.loads("{\"course_models.py\":\"\\\"\\\"\\\"Teaching adapters. Original equations/parameters remain in course_source.\\n\\nTime: h; concentration: g/L; volume: L; feed: L/h.\\nThe event/phase handling here corrects integration across feed discontinuities.\\n\\\"\\\"\\\"\\nimport random\\nimport numpy as np\\nfrom scipy.integrate import solve_ivp\\nfrom course_source.DAY3.parameters import Parameters as YeastParameters\\nfrom course_source.DAY3.model import initial_values, feed_values\\nfrom course_source.DAY3 import model as source_model\\n\\nSTATE_NAMES = ['Glucose', 'Xylose', 'Furfural', 'Furfuryl alcohol',\\n               '5-HMF', 'Acetic acid', 'Ethanol', 'Biomass', 'Volume']\\n\\ndef yeast_rhs(t, y, par, feed, feeding=False):\\n    # At t=0 the original Feed function is zero: recover reaction terms exactly.\\n    dy = source_model.fedbatch(0, y, par, feed)\\n    F = 0.1 + 0.005*(t-2) if feeding else 0.0\\n    dy[:8] += F * (np.asarray(feed[:8]) - y[:8]) / y[8]\\n    dy[8] = F\\n    return dy\\n\\ndef simulate_yeast(par=None, initial=None, end=50., method='BDF',\\n                   rtol=1e-7, atol=1e-9, feed=None):\\n    \\\"\\\"\\\"Integrate batch, feeding, and full-vessel phases on separate intervals.\\n\\n    Volume capacity is 2 L. The feed polynomial can be integrated analytically;\\n    split at the exact filling time instead of stepping over a state switch.\\n    \\\"\\\"\\\"\\n    from types import SimpleNamespace\\n    par = YeastParameters() if par is None else par\\n    y0 = np.asarray(initial_values() if initial is None else initial, dtype=float)\\n    if y0.shape != (9,) or not np.all(np.isfinite(y0)) or np.any(y0 < 0) or not 0 < y0[8] <= 2:\\n        raise ValueError('Use nine finite nonnegative states and 0 < V0 <= 2 L.')\\n    if not np.isfinite(end) or end <= 0:\\n        raise ValueError('end must be a finite positive time in hours.')\\n    feed = feed_values(y0.tolist()) if feed is None else list(feed)\\n    fill = 2 + (-0.1 + np.sqrt(0.1**2 + 0.01*(2-y0[8])))/0.005\\n    edges = sorted(set([0., float(end)] + [v for v in [2., fill] if 0 < v < end]))\\n    times, states, nfev = [], [], 0\\n    for a, b in zip(edges[:-1], edges[1:]):\\n        feeding = a >= 2 and a < fill and y0[8] < 2-1e-10\\n        grid = np.linspace(a, b, max(3, int((b-a)*12)+1))\\n        sol = solve_ivp(yeast_rhs, (a,b), y0, args=(par,feed,feeding),\\n                        method=method, t_eval=grid, rtol=rtol, atol=atol)\\n        if not sol.success:\\n            raise RuntimeError(sol.message)\\n        start = 1 if times else 0\\n        times.extend(sol.t[start:])\\n        states.extend(sol.y[:, start:].T)\\n        y0 = sol.y[:, -1].copy()\\n        nfev += sol.nfev\\n    return SimpleNamespace(t=np.array(times), y=np.array(states).T,\\n                           success=True, nfev=nfev, fill_time=fill)\\n\\ndef variable_feedstock(seed):\\n    \\\"\\\"\\\"BDG cell 7 ranges, local RNG; preserves zero-width FA draw.\\\"\\\"\\\"\\n    rng = random.Random(seed)\\n    bounds = [(35,40),(18,28),(0.6,2),(0,0),(0.5,2),(2,3),(0.2,0.9)]\\n    return [rng.uniform(lo,hi) for lo,hi in bounds] + [5.,0.7]\\n\\ndef variable_growth_parameters(seed):\\n    \\\"\\\"\\\"BDG cell 9: six independent +2 to +5% changes (one-sided).\\\"\\\"\\\"\\n    rng = random.Random(seed)\\n    par = YeastParameters()\\n    for name, baseline in [('numaxG',1.7271),('numaxX',1.6222),\\n                           ('YPSg',0.42),('YPSx',0.24),('YXSg',0.115),('YXSx',0.162)]:\\n        setattr(par, name, baseline*(1+rng.uniform(0.02,0.05)))\\n    return par\\n\\nclass ChemostatParameters:\\n    \\\"\\\"\\\"Values from DAY2/Control_Exercise_2, cell 3; no substituted mu_max.\\\"\\\"\\\"\\n    Ks, Ko, Yxs = 0.1, 0.2e-3, 0.4\\n    A, B, Eg, Gd, R, T = 1e10, 3e90, 58., 550., 8.314e-3, 293.15\\n    kla, Sf, V, F = 250., 10., 2., 0.5\\n    def __init__(self):\\n        self.mux = self.A*np.exp(-self.Eg/(self.R*self.T))/(1+self.B*np.exp(-self.Gd/(self.R*self.T)))\\n        self.O_sat = 0.21*0.027*np.exp(1142/self.T)*32/1000\\n\\ndef chemostat(t, y, par, kla=None):\\n    X, S, O = y\\n    mu = par.mux*S/(par.Ks+S)*O/(par.Ko+O)\\n    D = par.F/par.V\\n    transfer = par.kla if kla is None else kla\\n    return [ (mu-D)*X, D*(par.Sf-S)-mu*X/par.Yxs,\\n             transfer*(par.O_sat-O)-D*O-(1/par.Yxs-1)*mu*X ]\\n\\ndef oxygen_setpoint(t):\\n    return 0.003 if t > 10 else (0.006 if t > 5 else 0.005)\\n\\ndef oxygen_control(end=20., Kc=-50000., tauI=0.02):\\n    \\\"\\\"\\\"Original unsaturated PI; split at setpoint steps; returns actual actuation.\\\"\\\"\\\"\\n    from types import SimpleNamespace\\n    par = ChemostatParameters()\\n    if end <= 0 or tauI <= 0:\\n        raise ValueError('end and tauI must be positive.')\\n    y0 = [3.,0.,0.,0.]\\n    times, states = [], []\\n    edges = sorted(set([0., float(end)] + [v for v in [5.,10.] if v < end]))\\n    for a,b in zip(edges[:-1],edges[1:]):\\n        sp = oxygen_setpoint((a+b)/2)\\n        def rhs(t,y):\\n            X,S,O,I = y\\n            u = par.kla + Kc*(O-sp) + I\\n            return [*chemostat(t,[X,S,O],par,u), Kc/tauI*(O-sp)]\\n        sol = solve_ivp(rhs,(a,b),y0,method='BDF',t_eval=np.linspace(a,b,151),\\n                        rtol=1e-7,atol=[1e-9,1e-9,1e-11,1e-7])\\n        if not sol.success:\\n            raise RuntimeError(sol.message)\\n        start=1 if times else 0\\n        times.extend(sol.t[start:]); states.extend(sol.y[:,start:].T)\\n        y0=sol.y[:,-1]\\n    t=np.array(times); y=np.array(states).T\\n    sp=np.array([oxygen_setpoint(v) for v in t])\\n    u=par.kla + Kc*(y[2]-sp) + y[3]\\n    return SimpleNamespace(t=t,y=y,sp=sp,kla=u,success=True)\\n\",\"course_source/DAY3/model.py\":\"import numpy as np\\r\\n\\r\\ndef initial_values():\\r\\n    ## LIST OF INITIAL CONDITIONS.\\r\\n    # Name     Value     Index     Units\\r\\n    Glu0  =  39.7        # 1       g/L\\r\\n    Xyl0  =  23.5        # 2       g/L\\r\\n    Fur0  =  0.56        # 3       g/L\\r\\n    FA0   =  0           # 4       g/L\\r\\n    HMF0  =  0.2         # 5       g/L\\r\\n    HAc0  =  3.05        # 6       g/L\\r\\n    # Ac0   =  0.0001;    # 7       g/L\\r\\n    EtOH0 =  0.62        # 7       g/L\\r\\n    X0    =  1.75        # 8       g/L\\r\\n    V0    = 0.7          # 9       g/L\\r\\n    \\r\\n    init = [Glu0, Xyl0, Fur0, FA0, HMF0, HAc0, EtOH0, X0, V0]\\r\\n    return init\\r\\n\\r\\ndef feed_values(ini):\\r\\n    feed = ini.copy()\\r\\n    feed[7] = 0             # Xf = 0 !  \\r\\n    return feed\\r\\n\\r\\ndef Feed(t, V):\\r\\n    if t<2 or V>=2:\\r\\n        F = 0\\r\\n    else:\\r\\n        F = 0.1+0.005*(t-2)\\r\\n    return F\\r\\n\\r\\ndef fedbatch(t,x,par, Cfeed):\\r\\n    #  The following 8 components are considered in the model: \\r\\n    #  0. Glucose\\r\\n    #  1. Xylose\\r\\n    #  2. Furfural\\r\\n    #  3. Furfuryl alcohol\\r\\n    #  4. 5-HMF\\r\\n    #  5. HAc\\r\\n    #  6. Ethanol\\r\\n    #  7. Biomass\\r\\n    #  8. Tank volume\\r\\n\\r\\n    #  According to the kinetic model, the following chemical equations describe\\r\\n    #  the system. \\r\\n\\r\\n    #  1. Glucose  -----> Ethanol + Biomass\\r\\n    #  2. Xylose   -----> Ethanol + Biomass\\r\\n    #  3. Furfural -----> Furfuryl alcohol \\r\\n    #  4. HMF      -----> Acetic acid\\r\\n    #  5. HAc     <-----> Ac- [optional - not used here]\\r\\n    #  6. HAc      -----> Biomass (maintenence) \\r\\n\\r\\n\\r\\n    ## 1. STOICHIOMETRIC MATRIX. \\r\\n    # The 8 components and 6 reactions are considered in this model. One could\\r\\n    # also expand the model with Ac-, but that is not implemented.\\r\\n\\r\\n    #                Glu  Xyl  Fur  FA           HMF  HAc           EtOH        X  \\r\\n    st = np.array([[-1,   0,   0,   0,           0,   0,            par.YPSg,   par.YXSg],      # Glucose uptake\\r\\n                   [ 0,  -1,   0,   0,           0,   0,            par.YPSx,   par.YXSx],      # Xylose uptake\\r\\n                   [ 0,   0,  -1,   par.Y_FA_Fur,0,   0,            0,          0       ],      # Furfural uptake\\r\\n                   [ 0,   0,   0,   0,          -1,   par.Y_HAc_HMF,0,          0       ],      # HMF uptake\\r\\n                   [ 0,   0,   0,   0,           0,  -1,            0,          0       ]])     # HAc uptake\\r\\n\\r\\n\\r\\n\\r\\n\\r\\n    ## 2. DEFINE INDIVIDUAL KINETICS AND INHIBITION TERMS. \\r\\n    # Ph stands for phenomena, and it includes all the phenomena considered in\\r\\n    # the model, including reaction, inhibition and equilibria. Each of the\\r\\n    # phenomena is implemented separately below, for clarity\\r\\n    # 1. GLUCOSE UPTAKE RATE.\\r\\n    # numaxG: max consumption rate of glucose           s^-1       [1]\\r\\n    # KSPG:   affinity constant glucose                 g/L        [1]\\r\\n    # KiPG:   inhibition constant glucose               g/L        [1]\\r\\n    Ph1 = par.numaxG * x[0] * x[7] / (par.KSPG + x[0] + (x[0]**2)/par.KiPG)\\r\\n    Ph1 = max(0,Ph1)\\r\\n    # 2. XYLOSE UPTAKE RATE.\\r\\n    # numaxX: max consumption rate of xylose            s^-1       [1]\\r\\n    # KSPX:   affinity constant xylose                  g/L        [1]\\r\\n    # KiPX:   inhibition constant xylose                g/L        [1]\\r\\n    Ph2 = par.numaxX * x[1] * x[7] / (par.KSPX + x[1] + (x[1]**2)/par.KiPX)\\r\\n    Ph2 = max(0,Ph2)\\r\\n\\r\\n    # 3. FURFURAL UPTAKE RATE.\\r\\n    # numaxFur: max uptake rate of furfural               s^-1       [2]\\r\\n    # KSFur:    affinity constant furfural                g/L        [2]\\r\\n    Ph3 = par.numaxFur * x[2] * x[7] / (par.KSFur + x[2])\\r\\n    Ph3 = max(0,Ph3)\\r\\n\\r\\n    # 4. FURFURAL CONVERSION INTO FURFURYL ALCOHOL.\\r\\n    # Y_FA_Fur:  yield coefficient FA/Fur                  gFA/gFur   [2]\\r\\n    Ph4 = Ph3 * par.Y_FA_Fur\\r\\n    Ph4 = max(0,Ph4)\\r\\n\\r\\n    # 5. FURFURYL ALCOHOL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiFAg:    inhibition constant of FA on Glu uptake   g/L        [2]\\r\\n    Ph5 = 1 / (1 + x[3]/par.KiFAg)\\r\\n    Ph5 = max(0,Ph5)\\r\\n\\r\\n    # 6. FURFURYL ALCOHOL INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiFAx:    inhibition constant of FA on Xyl uptake   g/L        [2]\\r\\n    Ph6 = 1 / (1 + x[3]/par.KiFAx)\\r\\n    Ph6 = max(0,Ph6)\\r\\n\\r\\n    # 7. FURFURAL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiFurg:   inhibition constant of Fur on Glu uptake  g/L        [2]\\r\\n    Ph7 = 1 / (1 + x[2]/par.KiFurg)\\r\\n    Ph7 = max(0,Ph7)\\r\\n\\r\\n    # 8. FURFURAL INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiFurx:   inhibition constant of Fur on Xyl uptake  g/L        [2]\\r\\n    Ph8 = 1 / (1 + x[2]/par.KiFurx)\\r\\n    Ph8 = max(0,Ph8)\\r\\n\\r\\n    # 9. FURFURAL INHIBITS HMF UPTAKE RATE.\\r\\n    # KiFurHMF: inhibition constant of Fur on HMF         g/L        [2]\\r\\n    Ph9 = 1 / (1 + x[2]/par.KiFurHMF)\\r\\n    Ph9 = max(0,Ph9)\\r\\n\\r\\n    # 10. HMF INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiHMFg:   inhibition constant of HMF on Glu         g/L        [2]\\r\\n    Ph10 = 1 / (1 + x[4]/par.KiHMFg)\\r\\n    Ph10 = max(0,Ph10)\\r\\n\\r\\n    # 11. HMF INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiHMFx:   inhibition constant of HMF on Xyl         g/L        [2]\\r\\n    Ph11 = 1 / (1 + x[4]/par.KiHMFx)\\r\\n    Ph11 = max(0,Ph11)\\r\\n\\r\\n    # 12. HMF UPTAKE RATE. \\r\\n    # numaxHMF: max uptake rate of HMF                    s^-1       [2]\\r\\n    # KSHMF:    affinity constant HMF                     g/L        [2]\\r\\n    Ph12 = par.numaxHMF * x[7] * x[4] / (par.KSHMF + x[4])\\r\\n    Ph12 = max(0,Ph12)\\r\\n\\r\\n    #  13. pH INFLUENCES THE PAIR HAc/Ac.\\r\\n    # pH = 5.5;              % pH is controlled\\r\\n    # pKa = 4.75;            % pKa of acetic acid.\\r\\n    # Ph13 = x(7,1) / x(6,1) * 1 / (1 + 10^(-pH)/10^(-pKa));\\r\\n    # Ph13 = max(0,Ph13);\\r\\n\\r\\n\\r\\n    # 14. HAc UPTAKE RATE.\\r\\n    # numaxHAc: max uptake rate of HAc                    s^-1       [2]\\r\\n    # KSHAc:    affinity constant HAc                     g/L        [2]\\r\\n    Ph14 = x[7] * x[5] * par.numaxHAc / (par.KSHAc + x[5])\\r\\n    Ph14 = max(0,Ph14)\\r\\n\\r\\n\\r\\n    #15. HMF CONVERSION INTO HAc\\r\\n    #Y_HAc_HMF = p(21);     % yield coefficient HAc/HMF                 gHAc/gHMF  [2], [3]\\r\\n    #Ph15 = Y_HAc_HMF * Ph12;\\r\\n    #Ph15 = max(0,Ph15);\\r\\n\\r\\n    # 16. HAc INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiHAcg:   inhibition constant of HAc on Glu         g/L        [2]\\r\\n    Ph16 = 1 / (1 + x[5]/par.KiHAcg)\\r\\n    Ph16 = max(0,Ph16)\\r\\n\\r\\n    # 17. HAc INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiHAcx:   inhibition constant of HAc on Xyl         g/L         [2]\\r\\n    Ph17 = 1 / (1 + x[5]/par.KiHAcx)\\r\\n    Ph17 = max(0,Ph17)\\r\\n\\r\\n    #  18. PRODUCTION OF ETHANOL FORM GLUCOSE AND XYLOSE.\\r\\n    # YPSg:     yield ethanol-glucose                     gEtOH/gGlu [1]\\r\\n    # YPSx:     yield ethanol-xylose                      gEtOH/gGlu [1]\\r\\n    # Ph18a = par.YPSg * Ph1;\\r\\n    # Ph18b = par.YPSx * Ph2;\\r\\n    # Ph18a = max(0,Ph18a);\\r\\n    # Ph18b = max(0,Ph18b);\\r\\n\\r\\n    # 19. ETHANOL INHIBITS THE UPTAKE OF GLUCOSE AND XYLOSE.\\r\\n    # PMPg:     inhibition constant for glucose           g/L        [1]\\r\\n    # gammaG:   exponent factor inhibition glucose        g/L        [1]\\r\\n    # PMPx:     inhibition constant for xylose            g/L        [1]\\r\\n    # gammaX:   exponent factor inhibition xylose         g/L        [1]\\r\\n    Ph19a = 1-(x[6]/par.PMPg)**par.gammaG\\r\\n    Ph19b = 1-(x[6]/par.PMPx)**par.gammaX\\r\\n    Ph19a = max(0,Ph19a)\\r\\n    Ph19b = max(0,Ph19b)\\r\\n\\r\\n    #  20. CELL GROWTH.\\r\\n    # mGlu:     maintenance constant from glucose         g/L        [1]\\r\\n    # mXyl:     maintenance constant from xylose          g/L        [1]\\r\\n    # YXSg:     yield X-Glu                               gX/gGlu    [1]\\r\\n    # YXSx:     yield X-Xyl                               gX/gXyl    [1]\\r\\n    # mumaxG:   max growth of X from Glu                  h-1        [1]\\r\\n    # mumaxX:   max frowth of X from Xyl                  h-1        [1]\\r\\n\\r\\n    # Ph20a = max(0,(Ph1 + mGlu*x(8,1))*YXSg);\\r\\n    # Ph20b = max(0,(Ph1 + mXyl*x(8,1))*YXSx);\\r\\n\\r\\n    # 21. CATABOLITE REPRESSION.\\r\\n    # KiGlu:    inhibition constant of Glu to Xyl         g/L        Unknown\\r\\n    Ph21 = 1 / (1 + x[0]/par.KiGlu)\\r\\n\\r\\n    # 22. ACETATE IS USED FOR MAINTENANCE.\\r\\n    # mHAc:     maintenance constant from HAc              g/L        Unknown\\r\\n    # YXSHAc:   yield acetate biomass                      g/L        Unknown\\r\\n    Ph22 = max(0,par.mHAc * x[7] * par.YXSHAc)\\r\\n\\r\\n\\r\\n\\r\\n    ## 3. REACTION RATES. \\r\\n    # This includes the reaction rates with inhibition terms.\\r\\n    rates = np.zeros(5)\\r\\n    rates[0]  = Ph1 * Ph5 * Ph7 * Ph10 * Ph16 * Ph19a                            # Glucose uptake rate considering inhibitions.\\r\\n    rates[1]  = Ph2 * Ph6 * Ph8 * Ph11 * Ph17 * Ph19b * Ph21                     # Xylose uptake rate considering inhibitions.\\r\\n    rates[2]  = Ph3                                                              # Furfural uptake rate.\\r\\n    rates[3]  = Ph12 *  Ph9                                                      # 5-HMF uptake rate considering inhibitions.\\r\\n    rates[4]  = Ph14\\r\\n    \\r\\n    ## 4. MASS BALANCE FOR THE SYSTEM.\\r\\n    # Cfeed = feed_values()\\r\\n    dxdt = np.zeros(9)\\r\\n    r, c = st.shape\\r\\n    for i in range(c):\\r\\n        dxdt[i] = sum([rates[j] * st[j,i] for j in range(r)]) + Feed(t, x[8])*(Cfeed[i]-x[i])/x[8]        # Components mass balance -> FedBatch\\r\\n    dxdt[8] =  Feed(t, x[8])                 # Tank volume mass balance\\r\\n    return dxdt\\r\\n\\r\\nif __name__=='__main__':\\r\\n    from parameters import Parameters\\r\\n    init = initial_values()\\r\\n    par = Parameters()\\r\\n    dxdt = fedbatch(0, init, par)\\r\\n    print(dxdt)\\r\\n\\r\\n\\r\\n\\r\\n\",\"course_source/DAY3/parameters.py\":\"# [1] Krishnan et al, 1999\\r\\n# [2] Hanly et al, 2004\\r\\n# [3] Mauricio-Iglesias et al, \\r\\n\\r\\nclass Parameters():\\r\\n    '''Define all Parameters to be used in the model in a class'''\\r\\n    def __init__(self):\\r\\n        ## 1. GLUCOSE UPTAKE RATE.\\r\\n        # numaxG = 2.6808  ;       # max consumption rate of glucose (1.0005)         h^-1       [1]\\r\\n        self.numaxG = 1.7271          # max consumption rate of glucose (1.0005)  h^-1       [1]\\r\\n        # self.KSPG   = 0.14637         # affinity constant glucose to ethanol      g/L        [1]\\r\\n        self.KSPG   = 0.565         # affinity constant glucose to ethanol      g/L        [1]\\r\\n        # self.KiPG   = 4752.8          # inhibition constant glucose to ethanol    g/L        [1]\\r\\n        self.KiPG   = 4890          # inhibition constant glucose to ethanol    g/L        [1]\\r\\n        # p(1:3) = [numaxG, KSPG, KiPG];\\r\\n\\r\\n        ## 2. XYLOSE UPTAKE RATE.\\r\\n        #numaxX = 4.4851;         # max consumption rate of xylose            h^-1       [1]\\r\\n        # self.numaxX = 7.0531          # max consumption rate of xylose            h^-1       [1]\\r\\n        self.numaxX = 1.6222          # max consumption rate of xylose            h^-1       [1]\\r\\n        # self.KSPX   = 0.58763         # affinity constant xylose to ethanol       g/L        [1]\\r\\n        self.KSPX   = 3.4         # affinity constant xylose to ethanol       g/L        [1]\\r\\n        # self.KiPX   = 21.608          # inhibition constant xylose to ethanol     g/L        [1]\\r\\n        self.KiPX   = 18.1          # inhibition constant xylose to ethanol     g/L        [1]\\r\\n        # p(4:6) = [numaxX, KSPX, KiPX];\\r\\n\\r\\n        ## 3. FURFURAL UPTAKE RATE.\\r\\n        # self.numaxFur = 0.14309       # max uptake rate of furfural               h^-1       [2]\\r\\n        self.numaxFur = 4.67e-5*3600       # max uptake rate of furfural               h^-1       [2]\\r\\n        self.KSFur    = 0.05          # affinity constant furfural                g/L        [2]\\r\\n        #p(7:8) = [numaxFur, KSFur];\\r\\n\\r\\n        ## 4. FURFURAL CONVERSION INTO FURFURYL ALCOHOL.\\r\\n        self.Y_FA_Fur = 1.02          # yield coefficient FA/Fur                  gFA/gFur   [2]\\r\\n        #p(9) = Y_FA_Fur;\\r\\n        \\r\\n        ## 5. FURFURYL ALCOHOL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        self.KiFAg = 5                # inhibition constant of FA on Glu uptake   g/L        [2]3\\r\\n        #p(10) = KiFAg;\\r\\n\\r\\n        ## 6. FURFURYL ALCOHOL INHIBITS XYLOSE UPTAKE RATE.\\r\\n        self.KiFAx = 6                # inhibition constant of FA on Xyl uptake   g/L        [2]\\r\\n        #p(11) = KiFAx;\\r\\n\\r\\n        ## 7. FURFURAL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        # self.KiFurg = 0.4529          # inhibition constant of Fur on Glu uptake  g/L        [2]\\r\\n        self.KiFurg = 0.75            # inhibition constant of Fur on Glu uptake  g/L        [2]\\r\\n        #p(12) = KiFurg;\\r\\n\\r\\n        ## 8. FURFURAL INHIBITS XYLOSE UPTAKE RATE.\\r\\n        self.KiFurx = 0.35            # inhibition constant of Fur on Xyl uptake  g/L        [2]\\r\\n        #p(13) = KiFurx;\\r\\n\\r\\n        ## 9. FURFURAL INHIBITS HMF UPTAKE RATE.\\r\\n        self.KiFurHMF = 0.25          # inhibition constant of Fur on HMF         g/L        [2]\\r\\n        #p(14) = KiFurHMF;\\r\\n\\r\\n        ## 10. HMF INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        self.KiHMFg = 2               # inhibition constant of HMF on Glu         g/L        [2]\\r\\n        #p(15) = KiHMFg;\\r\\n\\r\\n        ## 11. HMF INHIBITS XYLOSE UPTAKE RATE.\\r\\n        self.KiHMFx = 10              # inhibition constant of HMF on Xyl         g/L        [2]\\r\\n        #p(16) = KiHMFx;\\r\\n\\r\\n        ## 12. HMF UPTAKE RATE. \\r\\n        # self.numaxHMF = 0.3154        # max uptake rate of HMF                    h^-1       [2]\\r\\n        self.numaxHMF = 8.76E-5*3600        # max uptake rate of HMF                    h^-1       [2]\\r\\n        self.KSHMF = 0.5              # affinity constant HMF                     g/L        [2]\\r\\n        #p(17:18) = [numaxHMF, KSHMF];\\r\\n\\r\\n        ## 13. pH INFLUENCES THE PAIR HAc/Ac.\\r\\n        self.pH = 5.5                 # pH is controlled\\r\\n        self.pKa = 4.75               # pKa of acetic acid.\\r\\n\\r\\n        ## 14. HAc UPTAKE RATE.\\r\\n        self.numaxHAc = 1.23E-5*3600*0.001     # max uptake rate of HAc                    h^-1       [2]\\r\\n        self.KSHAc = 2.5                 # affinity constant HAc                     g/L        [2]\\r\\n        #p(19:20) = [numaxHAc, KSHAc];\\r\\n\\r\\n        ## 15. HMF CONVERSION INTO HAc\\r\\n        self.Y_HAc_HMF = 0.534        # yield coefficient HAc/HMF                 gHAc/gHMF  [2], [3]\\r\\n        #p(21) = Y_HAc_HMF;\\r\\n\\r\\n        ## 16. HAc INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        # KiHAcg = 5.1827;         # inhibition constant of HAc on Glu         g/L        [2]\\r\\n        self.KiHAcg = 5.6703          # inhibition constant of HAc on Glu         g/L        [2]\\r\\n        # p(22) = KiHAcg;\\r\\n\\r\\n        ## 17. HAc INHIBITS XYLOSE UPTAKE RATE.\\r\\n        # KiHAcx = 0.73;           # inhibition constant of HAc on Xyl         g/L        [2]\\r\\n        self.KiHAcx = 3.8291          # inhibition constant of HAc on Xyl         g/L        [2]\\r\\n        # p(23) = KiHAcx;\\r\\n\\r\\n        ## 18. PRODUCTION OF ETHANOL FORM GLUCOSE AND XYLOSE.\\r\\n        # YPSg = 0.42;             # yield ethanol-glucose                     gEtOH/gGlu [1]\\r\\n        self.YPSg = 0.42              # yield ethanol-glucose                     gEtOH/gGlu [1]\\r\\n\\r\\n        self.YPSx = 0.24              # yield ethanol-xylose                      gEtOH/gGlu [1]\\r\\n        # p(24:25) = [YPSg, YPSx];\\r\\n\\r\\n        ## 19. ETHANOL INHIBITS THE UPTAKE OF GLUCOSE AND XYLOSE.\\r\\n        self.PMPg = 103             # inhibition constant for glucose           g/L        [1]\\r\\n        self.gammaG = 1.42            # exponent factor inhibition glucose        g/L        [1]\\r\\n        self.PMPx = 60.2           # inhibition constant for xylose            g/L        [1]\\r\\n        # PMPx = 20.9778;          # inhibition constant for xylose            g/L        [1]\\r\\n        self.gammaX =0.608          # exponent factor inhibition xylose         g/L        [1]\\r\\n        # gammaX = 1.7376;         # exponent factor inhibition xylose         g/L        [1]\\r\\n        # p(26:29) = [PMPg, gammaG, PMPx, gammaX];\\r\\n\\r\\n        ## 20. CELL GROWTH.\\r\\n        self.mGlu = 2.69E-5           # maintenance constant from glucose         g/L        [1]\\r\\n        self.mXyl = 1.86E-5           # maintenance constant from xylose          g/L        [1]\\r\\n        self.YXSg = 0.115            # yield X-Glu                               gX/gGlu    [1]\\r\\n        self.YXSx = 0.162             # yield X-Xyl                               gX/gXyl    [1]\\r\\n        self.mumaxG = 0.3308          # max growth of X from Glu                  h-1        [1]\\r\\n        self.mumaxX = 1.0008          # max frowth of X from Xyl                  h-1        [1]\\r\\n        # KSPGgr = 0.565;         # affinity constant glucose to growth       g/L        [1]\\r\\n        # KiPGgr = 283.7;         # inhibition constant glucose to growth     g/L        [1]\\r\\n        # KSPXgr = 3.4;           # affinity constant xylose to growth        g/L        [1]\\r\\n        # KiPXgr = 18.1;          # inhibition constant xylose to growth      g/L        [1]\\r\\n        # PMXg   = 95.4 ;         # max EtOH concentration for growth glucose g/L        [1]\\r\\n        # PMXx   = 59.04;         # max EtOH concentration for growth xylose  g/L        [1]\\r\\n        # p(30:35) = [mGlu, mXyl,YXSg, YXSx, mumaxG, mumaxX];\\r\\n\\r\\n        ## 21. CATABOLITE REPRESSION.\\r\\n        self.KiGlu = 13.763                # inhibition constant of Glu to Xyl         g/L        Unknown\\r\\n        # p(36) = KiGlu;\\r\\n\\r\\n        ## 22. ACETIC ACID IS USED FOR MAINTENANCE.\\r\\n        self.mHAc = 0                 # maintenance constant from HAc              g/L        Unknown\\r\\n        self.YXSHAc = 0               # yield acetate biomass                      g/L        Unknown \\r\\n        # p(38:39) = [mHAc, YXSHAc];\\r\\n\\r\\n\\r\\n        ## COLLECT PARAMETER NAMES\\r\\n        self.par_names = {'numaxG', 'KSPG', 'KiPG', 'numaxX', 'KSPX', 'KiPX', 'numaxFur', 'KSFur', 'Y_FA_Fur', 'KiFAg', 'KiFAx', 'KiFurg',\\r\\n            'KiFurx', 'KiFurHMF', 'KiHMFg', 'KiHMFx', 'numaxHMF', 'KSHMF', 'numaxHAc', 'KSHAc', 'Y_HAC_HMF', 'KiHAcg', 'KiHAcx', \\r\\n            'YPSg', 'YPSx', 'PMPg', 'gammaG', 'PMPx', 'gammaX', 'mGlu', 'mXyl', 'YXSg', 'YXSx', 'KiGlu'}\\r\\n\\r\\n        ## COLLECT DEPENDENT VARIABLE NAMES\\r\\n        self.var_names = ['Glucose', 'Xylose', 'Furfural','Furfuryl alcohol','5-HMF', 'Acetic acid', 'Ethanol', 'Biomass']\\r\\n\\r\\nif __name__=='__main__':\\r\\n    par = Parameters()\\r\\n    print(par.mHAc)\"}")
for _name, _content in _support.items():
    _path = Path(_name)
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_content, encoding='utf-8')
print('Model support ready. Requires numpy, scipy, matplotlib.')


## 01. A tank is an accumulation balance

**Objective:** Derive a volume balance, solve it numerically, and verify the result against a straight-line solution.

<p>For an incompressible liquid with no volume change on mixing, accumulation equals inflow minus outflow. The original example uses <code>Fi = 3</code>, <code>Fo = 2</code> m³/h and <code>V0 = 4</code> m³.</p><div class="equation">dV/dt = Fᵢ − Fₒ &nbsp; → &nbsp; V(t) = V₀ + (Fᵢ − Fₒ)t</div><p>The tank gains 1 m³ every hour. With no upper capacity constraint in this exercise, V(10) is 14 m³. An ODE solver does not know a vessel's capacity: if your model omits a physical boundary, the solver cannot impose it.</p><p>Use a parameter class or explicit function arguments to keep values out of the equation. These constant flows will become a manipulated outlet and disturbance inlet in the next exercise.</p>

**Task:** Complete the volume derivative, solve for 10 hours, and compare with the analytical trajectory. Then predict the slope when Fi = Fo before editing the parameters.

Source: DAY2/Control_Exercise_1.ipynb, original zero-based cells 3, 5, 6, 7, 8.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

Fi, Fo, V0 = 3., 2., 4.
def tank_rhs(t,y,Fi,Fo):
    return [Fi-Fo]

sol = solve_ivp(tank_rhs,(0,10),[V0],args=(Fi,Fo),t_eval=np.linspace(0,10,101))
exact = V0+(Fi-Fo)*sol.t
print("Final volume [m3]:",sol.y[0,-1])
plt.plot(sol.t,sol.y[0],label="Numerical"); plt.plot(sol.t,exact,"--",label="Balance")
plt.xlabel("Time [h]"); plt.ylabel("Volume [m³]"); plt.legend()


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert sol.success
assert np.allclose(tank_rhs(0,[4],3,2),[1]), 'Inflow adds volume; outflow removes it.'
assert np.allclose(sol.y[0],exact,atol=1e-8)
assert np.isclose(sol.y[0,-1],14)
print("All numerical checks passed.")


**Interpretation:** The state changes even though the derivative does not depend on V. This is an integrating process: a persistent flow imbalance causes unbounded volume drift in this ideal model.

**Hand in:** A volume balance, an analytical check, and the predicted response to balanced flows.


## 02. Close the loop with proportional control

**Objective:** Choose the correct feedback sign and quantify steady-state offset.

<p>The controlled variable is V; the manipulated variable is outlet flow. Follow the notebook convention <code>e = Vsp - V</code>. Increasing outlet flow decreases V, so the controller gain must be negative.</p><div class="equation">Fₒ = Fₒ₀ + Kc(Vsp − V)<br>dV/dt = Fᵢ − Fₒ</div><p>At steady state, inlet and outlet must match. Thus <code>Vss = Vsp - (Fi-Fo0)/Kc</code>. For <code>Kc=-1</code> h⁻¹, the controller settles at 5 m³ even though the setpoint is 4 m³. Proportional feedback requires a persistent error to generate the extra 1 m³/h of outlet flow.</p><p>The local dynamic equation has eigenvalue Kc. A positive Kc gives the wrong feedback sign for this outlet convention. No actuator saturation is included in the source model.</p>

**Task:** Implement the outlet controller. Solve the closed-loop balance, compute Vss from the steady-state equation, and plot both volume and outlet flow.

Source: DAY2/Control_Exercise_1.ipynb, original zero-based cells 9, 10, 12.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

Fi, Fo0, Vsp, Kc = 3., 2., 4., -1.
def rhs(t,y):
    Fo = Fo0+Kc*(Vsp-y[0])
    return [Fi-Fo]

sol = solve_ivp(rhs,(0,10),[4.],t_eval=np.linspace(0,10,201),rtol=1e-8,atol=1e-10)
Fo = Fo0+Kc*(Vsp-sol.y[0])
Vss = Vsp-(Fi-Fo0)/Kc
print("Predicted steady volume:",Vss,"Simulated:",sol.y[0,-1])
fig,ax=plt.subplots(2,1,sharex=True)
ax[0].plot(sol.t,sol.y[0]); ax[0].axhline(Vsp,ls="--",color="gray"); ax[0].set_ylabel("V [m³]")
ax[1].plot(sol.t,Fo); ax[1].set(xlabel="Time [h]",ylabel="Outlet [m³/h]")
fig.tight_layout()


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert np.allclose(rhs(0,[5.]),[0.]), 'At V=5, outlet must equal inlet.'
assert abs(sol.y[0,-1]-5) < 1e-3
assert abs(Fo[-1]-3) < 1e-3
print("All numerical checks passed.")


**Interpretation:** A stable controller can still miss its setpoint. Increasing |Kc| reduces offset but may demand stronger actuation; integral action addresses the offset directly.

**Hand in:** The controlled response and an algebraic prediction of its nonzero offset.


## 03. Integral action is another ODE

**Objective:** Represent a PI controller as an extra state and explain why it removes offset.

<p>Store the accumulated controller contribution as state I with units m³/h. The notebook adds I directly to the proportional action, so its derivative includes the gain.</p><div class="equation">Fₒ = Fₒ₀ + Kc(Vsp − V) + I<br>dI/dt = (Kc/τI)(Vsp − V)</div><p>The state vector is now <code>[V,I]</code>; its derivatives must appear in the same order. The original values are <code>Kc=-1</code> h⁻¹, <code>tauI=1</code> h, and <code>I0=0</code> m³/h. At steady state dI/dt=0 forces V=Vsp, and I supplies the missing outlet flow: I=1 m³/h.</p><p>This is an ideal unsaturated PI controller. Physical valves have bounds; adding clipping without anti-windup can make the integral grow while the actuator is saturated. Treat that as an extension, not an assumption hidden in this model.</p>

**Task:** Complete both derivatives. Run for 20 h, compare final V and I to their steady-state predictions, and plot the manipulated outlet alongside volume.

Source: DAY2/Control_Exercise_1.ipynb, original zero-based cells 11, 12.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

Fi, Fo0, Vsp, Kc, tauI = 3.,2.,4.,-1.,1.
def rhs(t,y):
    V,I=y
    e=Vsp-V
    Fo=Fo0+Kc*e+I
    return [Fi-Fo, Kc/tauI*e]

sol=solve_ivp(rhs,(0,20),[4.,0.],method="BDF",t_eval=np.linspace(0,20,301),rtol=1e-8,atol=1e-10)
Fo=Fo0+Kc*(Vsp-sol.y[0])+sol.y[1]
print("Final V [m3], I [m3/h]:",sol.y[:,-1])
fig,ax=plt.subplots(2,1,sharex=True)
ax[0].plot(sol.t,sol.y[0]); ax[0].axhline(Vsp,ls="--",color="gray"); ax[0].set_ylabel("V [m³]")
ax[1].plot(sol.t,Fo); ax[1].set(xlabel="Time [h]",ylabel="Outlet [m³/h]")
fig.tight_layout()


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert sol.success
assert np.allclose(rhs(0,[5.,0.]),[0.,1.]), 'The integral derivative includes Kc/tauI.'
assert abs(sol.y[0,-1]-4) < 1e-3, 'PI should remove the volume offset.'
assert abs(sol.y[1,-1]-1) < 1e-3, 'Integral action supplies the missing flow.'
print("All numerical checks passed.")


**Interpretation:** The small overshoot is a dynamic consequence of the two-state feedback system. The steady integral value is an actuator contribution, not an integrated volume.

**Hand in:** Overlay open-loop, P, and PI volumes using your earlier code; explain the final integral value using the flow balance.


## 04. A continuous fermentation with oxygen limitation

**Objective:** Relate biomass, substrate, and dissolved oxygen balances to dilution, reaction, and mass transfer.

<p>A well-mixed chemostat has equal inlet and outlet flows and constant volume. Its states are biomass X, substrate S, and oxygen O, all in g/L. Dilution is <code>D=F/V=0.25</code> h⁻¹, and the feed contains <code>Sf=10</code> g/L with no biomass or dissolved oxygen in this model.</p><div class="equation">μ = μx · S/(Ks + S) · O/(Ko + O)<br>dX/dt = (μ − D)X<br>dS/dt = D(Sf − S) − μX/Yxs<br>dO/dt = kLa(O* − O) − DO − (1/Yxs − 1)μX</div><p>The helper uses the notebook's temperature-dependent <code>mux</code> and computed <code>O_sat</code>, not its unused <code>mu_max=2.16</code> or <code>Osat=0.008</code>. At 293.15 K, <code>O_sat ≈ 0.00893</code> g/L. <code>Ks=0.1</code>, <code>Ko=0.0002</code> g/L, <code>Yxs=0.4</code> g/g, and <code>kLa=250</code> h⁻¹.</p><p>The oxygen-transfer time scale is roughly 1/kLa = 0.004 h, much faster than the 4 h residence time. This scale separation motivates an implicit solver and a small oxygen absolute tolerance.</p>

**Task:** Complete the biomass derivative in your own implementation. Compare it with the source-derived reference at a test state, then solve from [3,0,0] for 20 h and plot all three concentrations.

Source: DAY2/Control_Exercise_2.ipynb, original zero-based cells 2, 3, 4, 5, 6, 7.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from course_models import ChemostatParameters, chemostat
par=ChemostatParameters()
def rhs(t,y,par):
    X,S,O=y
    D=par.F/par.V
    mu=par.mux*S/(par.Ks+S)*O/(par.Ko+O)
    dX=(mu-D)*X
    dS=D*(par.Sf-S)-mu*X/par.Yxs
    dO=par.kla*(par.O_sat-O)-D*O-(1/par.Yxs-1)*mu*X
    return [dX,dS,dO]

sol=solve_ivp(rhs,(0,20),[3.,0.,0.],args=(par,),method="BDF",
              t_eval=np.linspace(0,20,401),rtol=1e-7,atol=[1e-9,1e-9,1e-11])
print("mux [1/h]:",par.mux,"O_sat [g/L]:",par.O_sat)
fig,ax=plt.subplots(3,1,sharex=True)
for i,name in enumerate(["Biomass","Substrate","Oxygen"]):
    ax[i].plot(sol.t,sol.y[i]); ax[i].set_ylabel(name+" [g/L]")
ax[-1].set_xlabel("Time [h]"); fig.tight_layout()


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert sol.success, sol.message
assert np.allclose(rhs(0,[2.,1.,0.004],par),chemostat(0,[2.,1.,0.004],par)), 'Verify all three balances.'
assert np.min(sol.y)>-1e-7, 'Concentrations must stay nonnegative within numerical tolerance.'
assert np.max(sol.y[2])<=par.O_sat+1e-7
assert np.isclose(par.F/par.V,0.25)
print("All numerical checks passed.")


**Interpretation:** Biomass initially declines because there is dilution before substrate/oxygen support growth. The oxygen consumption coefficient is the simplified relation used in the original model; it is not a universal yield law.

**Hand in:** Annotated species balances, three labelled trajectories, and a comparison of oxygen-transfer and residence times.


## 05. Control dissolved oxygen through kLa

**Objective:** Simulate an oxygen PI controller and reconstruct the actual actuator signal consistently.

<p>Use the notebook's oxygen-error convention <code>O - Osp</code>. Increasing kLa raises oxygen, so its gain is negative with this error definition. The setpoint changes from 0.005 to 0.006 g/L after 5 h, then to 0.003 g/L after 10 h.</p><div class="equation">kLa = 250 + Kc(O − Osp) + I<br>dI/dt = (Kc/τI)(O − Osp)</div><p>The original gain is <code>Kc=-50000</code> L/(g·h), and <code>tauI=0.02</code> h. The fourth state is the integral contribution in h⁻¹. The adapter integrates separate intervals at 5 and 10 h, keeping all states continuous at each setpoint jump.</p><p><strong>Source correction:</strong> the original actuator plot uses −60000 and omits I, even though the simulation uses −50000 and includes I. Always reconstruct the actuator with exactly the equation and parameters used in the derivative. This source model has no kLa limits or anti-windup.</p>

**Task:** Run the source-derived closed loop. Replace the placeholder actuator expression with the complete PI law. Plot oxygen/setpoint and kLa. Then halve the gain magnitude and compare tracking and actuation.

Source: DAY2/Control_Exercise_2.ipynb, original zero-based cells 8, 9, 10, 11.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from course_models import oxygen_control
Kc, tauI = -50000., 0.02
sol=oxygen_control(Kc=Kc,tauI=tauI)
actual_kla=250+Kc*(sol.y[2]-sol.sp)+sol.y[3]
print("Final oxygen [g/L]:",sol.y[2,-1])
print("kLa range [1/h]:",actual_kla.min(),actual_kla.max())
fig,ax=plt.subplots(2,1,sharex=True)
ax[0].plot(sol.t,sol.y[2],label="Oxygen"); ax[0].plot(sol.t,sol.sp,"--",label="Setpoint")
ax[0].set_ylabel("O [g/L]"); ax[0].legend()
ax[1].plot(sol.t,actual_kla); ax[1].set(xlabel="Time [h]",ylabel="kLa [1/h]")
fig.tight_layout()


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert np.allclose(actual_kla,sol.kla), 'Include the proportional AND integral terms with the simulated gain.'
assert abs(sol.y[2,-1]-0.003)<2e-5, 'Check final setpoint tracking.'
assert np.all(np.isfinite(actual_kla))
print("All numerical checks passed.")


**Interpretation:** The actuator can jump when the setpoint changes, while oxygen and the integral state remain continuous. Evaluate both tracking and actuator demands before judging a controller.

**Hand in:** A correctly reconstructed actuator plot, tracking comparison for two gains, and a discussion of physical kLa limits.


## 06. Day 2 lab: a solver you can defend

**Objective:** Use solver diagnostics and tolerance refinement to justify a stiff-system simulation.

<p>The chemostat couples fast oxygen transfer with slower growth and dilution. Compare methods on the <em>same equations, initial states, tolerances, and requested output times</em>. A solver comparison is meaningless if each method solves a different model.</p><div class="equation">εⱼ = maxₜ |yⱼ,test(t) − yⱼ,reference(t)|</div><p>Use BDF with <code>rtol=1e-9</code> and state-specific absolute tolerances as a refined numerical reference. This is a convergence check, not proof that the physical model is correct. The reference should be independently checked using balance residuals and physical bounds.</p><p>RK45 can solve a stiff problem but may take many small steps. Record <code>nfev</code>, <code>njev</code>, and <code>nlu</code> when available. Implicit methods pay for Jacobian calculations and linear solves, so function evaluations are only one part of their cost.</p>

**Task:** Add a Radau comparison to the existing RK45 and BDF runs. Record statewise deviations from the refined reference and compare function evaluations. Explain which solver you would use for a longer study.

Source: DAY2/Control_Exercise_2.ipynb, original zero-based cells 3, 5, 6, 9, 10.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from course_models import ChemostatParameters,chemostat
par=ChemostatParameters()
t=np.linspace(0,20,201)
ref=solve_ivp(chemostat,(0,20),[3.,0.,0.],args=(par,),method="BDF",t_eval=t,
              rtol=1e-9,atol=[1e-11,1e-11,1e-13])
results={}
methods=["RK45","BDF","Radau"]
for method in methods:
    sol=solve_ivp(chemostat,(0,20),[3.,0.,0.],args=(par,),method=method,t_eval=t,
                  rtol=1e-6,atol=[1e-8,1e-8,1e-10])
    if not sol.success:
        raise RuntimeError(sol.message)
    results[method]=np.max(abs(sol.y-ref.y),axis=1)
    print(method,"state errors:",results[method],"nfev:",sol.nfev,"nlu:",sol.nlu)


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert ref.success
assert set(results)=={'RK45','BDF','Radau'}, 'Compare all three methods.'
assert all(np.max(e[:2])<2e-4 and e[2]<2e-6 for e in results.values()), 'Solutions should agree under refinement.'
print("All numerical checks passed.")


**Interpretation:** Solver agreement verifies numerical implementation under refinement. It does not validate yields, kinetic laws, or biological assumptions against experimental data.

**Hand in:** Submit the solver comparison and an explanation of stiffness, absolute tolerances, controller sign conventions, and state order.
